# SSBG Data Exploration

In [1]:
import pandas as pd

df = pd.read_pickle('../data/interim/ssbg_data_cleaned.pkl')

df.info()
df.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21240 entries, 0 to 21239
Data columns (total 15 columns):
 #   Column                           Non-Null Count  Dtype   
---  ------                           --------------  -----   
 0   year                             21240 non-null  category
 1   state_name                       21240 non-null  category
 2   line_num                         21240 non-null  category
 3   service_category                 21240 non-null  category
 4   ssbg_expenditures                21240 non-null  int64   
 5   tanf_transfer_funds              21240 non-null  int64   
 6   total_ssbg_expenditures          21240 non-null  int64   
 7   other_fed_state_and_local_funds  21240 non-null  int64   
 8   total_expenditures               21240 non-null  int64   
 9   children                         21240 non-null  int64   
 10  adults_59_and_younger            21240 non-null  int64   
 11  adults_60_and_older              21240 non-null  int64   
 12  adul

,year,state_name,line_num,service_category,ssbg_expenditures,tanf_transfer_funds,total_ssbg_expenditures,other_fed_state_and_local_funds,total_expenditures,children,adults_59_and_younger,adults_60_and_older,adults_unknown,total_adults,total_recipients
count,21240.0,21240,21240.0,21240,2.124000e+04,2.124000e+04,2.124000e+04,2.124000e+04,2.124000e+04,2.124000e+04,2.124000e+04,21240.000000,2.124000e+04,2.124000e+04,2.124000e+04
unique,13.0,57,30.0,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,2017.0,Alabama,1.0,Administrative Costs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1710.0,390,708.0,708,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,9.866400e+05,7.002008e+05,1.686841e+06,1.791365e+07,1.960049e+07,7.094680e+03,2.990019e+03,1141.829896,4.296867e+03,8.428716e+03,1.552340e+04
std,NaN,NaN,NaN,NaN,4.993519e+06,6.938445e+06,9.920535e+06,1.862760e+08,1.922902e+08,7.553702e+04,5.387372e+04,14359.904396,1.065254e+05,1.258507e+05,1.739444e+05
min,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00
25%,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00
50%,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00
75%,NaN,NaN,NaN,NaN,1.599470e+05,0.000000e+00,2.487748e+05,5.564750e+04,9.507055e+05,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,6.500000e+02


## Category Columns
From the cleaning process we know that two of the categorical columns have non-uniform value_counts. [year and state_name]

In [2]:
category_cols = [df[col].dtype == 'category' for col in df.columns]
# examine the value counts for each category column after conversion and list any where the counts are not all the same
cols_not_uniform_value_counts = []
for col in df.columns[category_cols]:
    value_counts = df[col].value_counts()
    if value_counts.nunique() > 1:
        cols_not_uniform_value_counts.append(col)

print("Columns with non-uniform value counts:", cols_not_uniform_value_counts)

#Show the value counts and highlight any values that are lower than the max
for col in cols_not_uniform_value_counts:
    print(f"Value counts for '{col}':")
    value_counts = df[col].value_counts().sort_index()
    max_count = value_counts.max()
    print(f"Max count for '{col}': {max_count}")
    for category, count in value_counts.items():
        if count < max_count: 
            print(f"  {category}: {count}")
    print()

Columns with non-uniform value counts: ['year', 'state_name']
Value counts for 'year':
Max count for 'year': 1710
  2010: 1560
  2011: 1560
  2012: 1560
  2013: 1560
  2014: 1560
  2015: 1560
  2016: 1620

Value counts for 'state_name':
Max count for 'state_name': 390
  American Samoa: 180
  Guam: 180
  Massachusetts Commission for the Blind: 180
  Northern Mariana Islands: 210
  U.S. Virgin Islands: 210



Figure out when the territories and MassCFB started reporting data to the Annual Report dataset.  

In [3]:
# filter to just the state_names we want ['Massachusetts Commission for the Blind', 'American Samoa', 'Guam', 'Northern Mariana Islands', 'U.S. Virgin Islands']
state_names = ['Massachusetts Commission for the Blind', 'American Samoa', 'Guam', 'Northern Mariana Islands', 'U.S. Virgin Islands']
df_filtered = df[df['state_name'].isin(state_names)]


print("First year with more than zero total_ssbg_expenditures for each territory (and MassCFB):")
for state in state_names:
    df_state = df_filtered[df_filtered['state_name'] == state]
    # print the first year with more than zero summed total_ssbg_expenditures when grouped by year
    years_with_data = df_state.groupby('year').total_ssbg_expenditures.sum()
    first_year_with_data = years_with_data.loc[lambda x: x > 0]
    if len(first_year_with_data) > 0:
        print(f"{state}: {first_year_with_data.index[0]}")

First year with more than zero total_ssbg_expenditures for each territory (and MassCFB):
Massachusetts Commission for the Blind: 2017
American Samoa: 2017
Guam: 2017
Northern Mariana Islands: 2016
U.S. Virgin Islands: 2016


## Massachusetts Commission for the Blind 'Explanation'

In [4]:
# average total_ssbg_expenditures for Massachusetts Commission for the Blind across years where it was reporting data (2017-2022)
df_mass_cfb = df[(df['state_name'] == 'Massachusetts Commission for the Blind') & (df['year'].isin([2017, 2018, 2019, 2020, 2021, 2022]))]
df_mass_cfb.groupby('year').total_ssbg_expenditures.sum()

year
2010         0
2011         0
2012         0
2013         0
2014         0
2015         0
2016         0
2017    665390
2018    665701
2019    667910
2020    670046
2021    672623
2022    676223
Name: total_ssbg_expenditures, dtype: int64

In [5]:
# show rows withmore than 0 total_ssbg_expenditures for Massachusetts Commission for the Blind
df_mass_cfb = df_mass_cfb[df_mass_cfb['total_ssbg_expenditures'] > 0]
df_mass_cfb.service_category.value_counts().sort_values(ascending=False)

service_category
Special Services - Disabled                 6
Administrative Costs                        0
Case Management                             0
Congregate Meals                            0
Counseling Services                         0
Day Care - Adults                           0
Day Care - Children                         0
Education and Training Services             0
Employment Services                         0
Family Planning Services                    0
Foster Care Services - Adults               0
Foster Care Services - Children             0
Health-Related Services                     0
Home-Based Services                         0
Home-Delivered Meals                        0
Housing Services                            0
Independent/Transitional Living Services    0
Information and Referral                    0
Legal Services                              0
Other Services                              0
Pregnancy and Parenting                     0
Prevention and In

In [6]:
# find the total_ssbg_expenditures for Massachusetts row Special Services - Disabled
df[(df['service_category'] == 'Special Services - Disabled') & (df['state_name'] == 'Massachusetts')]

,year,state_name,line_num,service_category,ssbg_expenditures,tanf_transfer_funds,total_ssbg_expenditures,other_fed_state_and_local_funds,total_expenditures,children,adults_59_and_younger,adults_60_and_older,adults_unknown,total_adults,total_recipients
654,2010,Massachusetts,25,Special Services - Disabled,722549,0,722549,3576492,4299041,796,0,0,3822,3822,4618
2214,2011,Massachusetts,25,Special Services - Disabled,726144,0,726144,3821831,4547975,758,0,0,4593,4593,5351
3774,2012,Massachusetts,25,Special Services - Disabled,717016,0,717016,3801921,4518937,776,0,0,4789,4789,5565
5334,2013,Massachusetts,25,Special Services - Disabled,678342,0,678342,5015191,5693533,751,0,0,4013,4013,4764
6894,2014,Massachusetts,25,Special Services - Disabled,664282,0,664282,3244481,3908763,743,0,0,4287,4287,5030
8454,2015,Massachusetts,25,Special Services - Disabled,663545,0,663545,3406807,4070352,896,0,0,4387,4387,5283
10014,2016,Massachusetts,25,Special Services - Disabled,666613,0,666613,3148334,3814947,659,0,0,4738,4738,5397
11694,2017,Massachusetts,25,Special Services - Disabled,0,0,0,0,0,0,0,0,0,0,0
13404,2018,Massachusetts,25,Special Services - Disabled,0,0,0,0,0,0,0,0,0,0,0
15114,2019,Massachusetts,25,Special Services - Disabled,0,0,0,0,0,0,0,0,0,0,0


Notes: MassCFB expenditures are basically pull outs from Massachusetts state expenditures in previous years in the category Special Services - Disabled.   
They should probably be combined with the Massachusetts state expenditures for analysis purposes.

## Yearly Averages

In [7]:
# sum total_ssbg_expenditures by year
total_expenditures_by_year = df.total_ssbg_expenditures.groupby(df.year).sum()
# add column for percent change from previous year
total_expenditures_by_year_df = total_expenditures_by_year.to_frame().reset_index()
total_expenditures_by_year_df['percent_change'] = (total_expenditures_by_year_df.total_ssbg_expenditures.pct_change() * 100).round(2)

# print df with formatted column total_ssbg_expenditures with commas for thousands separator and a dollar sign
print(total_expenditures_by_year_df.to_string(index=False, formatters={'total_ssbg_expenditures': '${:,.0f}'.format}))


year total_ssbg_expenditures  percent_change
2010          $2,832,195,426             NaN
2011          $2,752,726,719           -2.81
2012          $2,802,044,562            1.79
2013          $2,961,541,885            5.69
2014          $2,735,689,104           -7.63
2015          $2,761,559,075            0.95
2016          $2,766,318,653            0.17
2017          $2,522,772,432           -8.80
2018          $2,622,416,486            3.95
2019          $2,884,737,074           10.00
2020          $2,774,505,540           -3.82
2021          $2,747,557,590           -0.97
2022          $2,664,435,921           -3.03


In [8]:
# Group by year and srevice category and identify the top three service categories each year by total_ssbg_expenditures
grouped = df.groupby(['year', 'service_category']).total_ssbg_expenditures.sum()
top_three_by_year = grouped.groupby(level=0, group_keys=False).nlargest(3)
print("Top three service categories by total_ssbg_expenditures each year:")
print(top_three_by_year)

Top three service categories by total_ssbg_expenditures each year:
year  service_category               
2010  Foster Care Services - Children    376530833
      Day Care - Children                370717380
      Special Services - Disabled        344546853
2011  Foster Care Services - Children    354655411
      Day Care - Children                339089941
      Special Services - Disabled        330022082
2012  Foster Care Services - Children    394831135
      Protective Services - Children     331039784
      Special Services - Disabled        307580174
2013  Foster Care Services - Children    428918942
      Protective Services - Children     373396067
      Prevention and Intervention        312754821
2014  Foster Care Services - Children    426929948
      Protective Services - Children     328617452
      Day Care - Children                299759015
2015  Foster Care Services - Children    432323535
      Protective Services - Children     297350878
      Day Care - Children   